# Cleaning Dataset Hortikultura & Biofarmaka — Kabupaten Kediri

Notebook ini membersihkan 7 file secara **spesifik per file** (bukan generik), karena tiap file
punya struktur yang beda. Ditulis berdasarkan isi asli tiap file, bukan tebakan.

**File yang dibersihkan:**
| File | Struktur asli |
|---|---|
| `Luas_Panen_Biofarmaka_per_kecamatan.csv` | Wide: 1 baris = 1 kecamatan, kolom = tiap komoditas. Ada baris total "Kediri". |
| `Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Produksi_Hortikultura_Buah_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Produksi_Hortikultura_Sayur_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Luas_Panen_Hortikultura_Sayur.csv` | Long, tingkat **Kabupaten** saja (kolom: Jenis Tanaman, nilai) |
| `Total_Produksi_Hortikultura_Buah.csv` | Sama (long, tingkat Kabupaten) |
| `Total_Produksi_Hortikultura_Sayur.csv` | Sama (long, tingkat Kabupaten) |

**Temuan penting dari pengecekan isi file:**
1. **Datanya genuine, bukan estimasi** — baris "Kediri" di tiap file wide adalah total resmi, dan setelah
   dicek, **sama persis** dengan hasil jumlah 26 kecamatan. Ini jadi alat validasi otomatis di notebook ini.
2. **Kolom "Keji Beling" di file biofarmaka isinya `...` di semua baris** (bukan data, kemungkinan error input
   dari sumbernya). Notebook ini mendeteksi dan **membuang** komoditas ini secara eksplisit, bukan
   mengubahnya diam-diam jadi 0 atau NaN.
3. **Satuan tidak seragam** antar komoditas di beberapa file: biofarmaka campur **m²** dan **pohon**
   (artinya sebagian bukan luas area, tapi jumlah pohon!); luas panen sayur mayoritas **hektar** tapi
   jamur dalam **m²**. Notebook ini menyimpan kolom `satuan` eksplisit, tidak memaksakan satu satuan.
4. **Dua kolom dengan label satuan yang tidak konsisten** (teks bilang satu hal, kode singkatannya bilang
   lain) — `Serai` (tertulis "meter persegi" tapi kode satuan "Pohon") dan `Jamur tiram` di file luas panen
   sayur (tertulis "meter persegi" tapi kode satuan "Ha"). Notebook ini memberi peringatan, bukan menebak
   mana yang benar — perlu kamu cek ke sumber aslinya.
5. **Nama kecamatan "Kayen Kidul"** (pakai spasi) di semua file wide, sedangkan file lain yang kita proses
   sebelumnya pakai "Kayenkidul" (tanpa spasi). Notebook ini menyamakan ke "Kayenkidul".

**Catatan soal "-" (strip):** dalam data BPS, "-" pada tabel komoditas biasanya berarti **tidak ada
produksi/luas panen** (bukan data hilang), sehingga di sini "-" diperlakukan sebagai **0**, bukan kosong.
Ini asumsi standar BPS, tapi tetap perlu kamu sadari — beda dengan `...` yang diperlakukan sebagai data
tidak valid (bukan 0).

**Output** (kolom seminimal mungkin, sesuai hasil cleaning saja — tanpa kolom tanggal proses dll.):
- `data/processed/biofarmaka/luas_panen_biofarmaka_per_kecamatan.csv` → `kecamatan, komoditas, luas_panen, satuan`
- `data/processed/hortikultura/luas_panen_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, luas_panen, satuan`
- `data/processed/hortikultura/produksi_hortikultura_buah_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `data/processed/hortikultura/produksi_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `data/processed/hortikultura/luas_panen_hortikultura_sayur_kabupaten.csv` → `komoditas, luas_panen, satuan`
- `data/processed/hortikultura/total_produksi_hortikultura_buah_kabupaten.csv` → `komoditas, produksi_kuintal`
- `data/processed/hortikultura/total_produksi_hortikultura_sayur_kabupaten.csv` → `komoditas, produksi_kuintal`

In [ ]:
# 1. IMPORT LIBRARY

import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

In [ ]:
# 2. CARI LOKASI PROJECT (folder 'data') SECARA OTOMATIS

CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan dari direktori notebook ini ke atas. "
        "Pastikan notebook dijalankan dari dalam repository (mis. dari folder 'notebooks/')."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_HORTI = os.path.join(DATA_DIR, "raw", "hortikultura")
RAW_BIOFARMAKA = os.path.join(DATA_DIR, "raw", "biofarmaka")
OUT_HORTI = os.path.join(DATA_DIR, "processed", "hortikultura")
OUT_BIOFARMAKA = os.path.join(DATA_DIR, "processed", "biofarmaka")

for d in (OUT_HORTI, OUT_BIOFARMAKA):
    os.makedirs(d, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# 3. MASTER KECAMATAN (26 kecamatan baku) & KOREKSI NAMA

MASTER_KECAMATAN = [
    "Mojo", "Semen", "Ngadiluwih", "Kras", "Ringinrejo", "Kandat", "Wates", "Ngancar",
    "Plosoklaten", "Gurah", "Puncu", "Kepung", "Kandangan", "Pare", "Badas", "Kunjang",
    "Plemahan", "Purwoasri", "Papar", "Pagu", "Kayenkidul", "Gampengrejo", "Ngasem",
    "Banyakan", "Grogol", "Tarokan",
]

# Perbaikan nama yang ditemukan langsung di file (bukan tebakan/fuzzy, supaya aman & presisi).
KOREKSI_KECAMATAN = {
    "Kayen Kidul": "Kayenkidul",
}

print(f"✓ {len(MASTER_KECAMATAN)} kecamatan baku siap.")

In [ ]:
# 4. FUNGSI BANTU

def parse_angka(nilai):
    """'-' -> 0.0 (konvensi BPS: tidak ada produksi/luas panen).
    '...' atau kosong -> NaN (data tidak valid/tidak tersedia, BUKAN nol)."""
    if pd.isna(nilai):
        return np.nan
    s = str(nilai).strip()
    if s == "-":
        return 0.0
    if s in ("", "...", "nan"):
        return np.nan
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def standar_kecamatan(nama):
    nama = str(nama).strip()
    return KOREKSI_KECAMATAN.get(nama, nama)


POLA_KOLOM_WIDE = re.compile(r"^(?:Luas Panen|Produksi)\s+(.*?)\s*\(([^()]*)\)\s*\(([^()]*)\)$")


def urai_kolom_wide(nama_kolom):
    """'Luas Panen Jahe (meter persegi) (M2)' -> ('Jahe', 'meter persegi', 'M2').
    Return (None, None, None) kalau pola tidak cocok (supaya ketahuan, bukan diam-diam dilewati)."""
    m = POLA_KOLOM_WIDE.match(nama_kolom.strip())
    if not m:
        return None, None, None
    return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()


def urai_baris_kabupaten(teks):
    """'Bawang Daun/Scallion(ha)' -> ('Bawang Daun/Scallion', 'ha'). Menangani teks yang ada
    baris baru di dalamnya (field CSV multi-baris yang dikutip)."""
    teks = re.sub(r"\s+", " ", str(teks).strip())
    m = re.match(r"^(.*)\(([^()]+)\)$", teks)
    if not m:
        return teks, None
    return m.group(1).strip(), m.group(2).strip()


print("✓ Fungsi bantu siap.")

In [ ]:
# 5. FUNGSI: BERSIHKAN SATU FILE WIDE (kecamatan x komoditas, ada baris total 'Kediri')

def bersihkan_file_wide(path, nama_value, cek_satuan_konsisten=None):
    """nama_value: nama kolom nilai di output ('luas_panen' atau 'produksi_kuintal').
    cek_satuan_konsisten: satuan yang diharapkan sama untuk semua komoditas (mis. 'Kw'), atau
    None kalau satuan memang boleh campur (maka kolom 'satuan' ikut disimpan)."""

    df_raw = pd.read_csv(path, encoding="utf-8-sig")
    kolom_kecamatan = df_raw.columns[0]
    kolom_komoditas = [c for c in df_raw.columns[1:]]

    print(f"--- {os.path.basename(path)} ---")
    print(f"  {len(df_raw)} baris (termasuk baris total), {len(kolom_komoditas)} kolom komoditas.")

    # Pisahkan baris total Kabupaten ('Kediri') dari baris kecamatan.
    is_total = df_raw[kolom_kecamatan].astype(str).str.strip().eq("Kediri")
    if not is_total.any():
        print("  ⚠️  Baris total 'Kediri' tidak ditemukan — validasi sum dilewati.")
        baris_total = None
    else:
        baris_total = df_raw.loc[is_total].iloc[0]
    df_kec = df_raw.loc[~is_total].copy()

    df_kec["kecamatan"] = df_kec[kolom_kecamatan].apply(standar_kecamatan)
    jumlah_unik = df_kec["kecamatan"].nunique()
    if jumlah_unik != len(MASTER_KECAMATAN):
        print(f"  ⚠️  Kecamatan unik = {jumlah_unik}, bukan {len(MASTER_KECAMATAN)}.")
    tidak_dikenal = set(df_kec["kecamatan"]) - set(MASTER_KECAMATAN)
    if tidak_dikenal:
        print(f"  ⚠️  Nama kecamatan tidak dikenali: {tidak_dikenal}")

    baris_keluar = []
    kolom_dibuang = []

    for kolom in kolom_komoditas:
        nama_komoditas, satuan_teks, satuan_kode = urai_kolom_wide(kolom)
        if nama_komoditas is None:
            print(f"  ⚠️  Nama kolom tidak dikenali polanya, dilewati: {kolom!r}")
            kolom_dibuang.append((kolom, "pola nama kolom tidak dikenali"))
            continue

        nilai_mentah = df_kec[kolom].astype(str)
        proporsi_invalid = nilai_mentah.eq("...").mean()
        if proporsi_invalid == 1.0:
            print(f"  ⚠️  Kolom '{nama_komoditas}' isinya '...' di semua baris (bukan data). Dibuang dari output.")
            kolom_dibuang.append((kolom, "seluruh nilai '...' (data tidak valid)"))
            continue
        elif proporsi_invalid > 0:
            print(f"  ⚠️  Kolom '{nama_komoditas}': {proporsi_invalid:.0%} baris berisi '...' (data tidak valid sebagian).")

        if cek_satuan_konsisten and satuan_kode != cek_satuan_konsisten:
            print(f"  ⚠️  Satuan '{nama_komoditas}' = {satuan_kode!r}, berbeda dari yang diharapkan ({cek_satuan_konsisten!r}).")

        satuan_m2_kata = "meter" in satuan_teks.lower()
        satuan_kode_m2 = satuan_kode.upper() == "M2"
        if satuan_m2_kata != satuan_kode_m2 and satuan_kode.lower() not in ("kw",):
            print(f"  ⚠️  Label satuan tidak konsisten untuk '{nama_komoditas}': teks={satuan_teks!r}, kode={satuan_kode!r}.")

        for _, baris in df_kec.iterrows():
            rec = {"kecamatan": baris["kecamatan"], "komoditas": nama_komoditas,
                  nama_value: parse_angka(baris[kolom])}
            if cek_satuan_konsisten is None:
                rec["satuan"] = satuan_kode
            baris_keluar.append(rec)

    df_long = pd.DataFrame(baris_keluar)

    # Validasi: total per komoditas (sum 26 kecamatan) vs baris 'Kediri'.
    if baris_total is not None:
        cek = []
        for kolom in kolom_komoditas:
            nama_komoditas, _, _ = urai_kolom_wide(kolom)
            if nama_komoditas is None or any(nama_komoditas == k for k, _ in kolom_dibuang):
                continue
            total_resmi = parse_angka(baris_total[kolom])
            total_hitung = df_long.loc[df_long.komoditas == nama_komoditas, nama_value].sum()
            if pd.notna(total_resmi) and not np.isclose(total_resmi, total_hitung, atol=0.01):
                cek.append((nama_komoditas, total_resmi, total_hitung))
        if cek:
            print(f"  ⚠️  {len(cek)} komoditas yang totalnya TIDAK cocok dengan baris 'Kediri':")
            for nk, tr, th in cek:
                print(f"      {nk}: resmi={tr}, hasil hitung={th}")
        else:
            print("  ✓ Semua total per komoditas cocok dengan baris 'Kediri'.")

    print(f"  ✓ Hasil akhir: {len(df_long)} baris ({df_long['komoditas'].nunique()} komoditas × {jumlah_unik} kecamatan).\n")
    return df_long


print("✓ Fungsi bersihkan_file_wide siap.")

In [ ]:
# 6. FUNGSI: BERSIHKAN FILE LONG TINGKAT KABUPATEN (Jenis Tanaman, nilai)

def bersihkan_file_kabupaten(path, nama_value):
    df_raw = pd.read_csv(path, encoding="utf-8-sig")
    kolom_nama, kolom_nilai = df_raw.columns[0], df_raw.columns[1]

    print(f"--- {os.path.basename(path)} ---")
    print(f"  {len(df_raw)} baris komoditas.")

    hasil = []
    for _, baris in df_raw.iterrows():
        nama_komoditas, satuan = urai_baris_kabupaten(baris[kolom_nama])
        rec = {"komoditas": nama_komoditas, nama_value: parse_angka(baris[kolom_nilai])}
        if satuan is not None:
            rec["satuan"] = satuan
        hasil.append(rec)

    df_out = pd.DataFrame(hasil)
    satuan_unik = df_out["satuan"].nunique() if "satuan" in df_out else 0
    if satuan_unik > 1:
        print(f"  ⚠️  Satuan tidak seragam: {sorted(df_out['satuan'].unique())}")

    print(f"  ✓ Hasil akhir: {len(df_out)} baris.\n")
    return df_out


print("✓ Fungsi bersihkan_file_kabupaten siap.")

In [ ]:
# 7. BIOFARMAKA — Luas Panen per Kecamatan

path = os.path.join(RAW_BIOFARMAKA, "Luas_Panen_Biofarmaka_per_kecamatan.csv")
df_biofarmaka = bersihkan_file_wide(path, nama_value="luas_panen", cek_satuan_konsisten=None)

display(df_biofarmaka.head(10))
df_biofarmaka.to_csv(os.path.join(OUT_BIOFARMAKA, "luas_panen_biofarmaka_per_kecamatan.csv"),
                     index=False, encoding="utf-8-sig")

In [ ]:
# 8. HORTIKULTURA SAYUR — Luas Panen per Kecamatan

path = os.path.join(RAW_HORTI, "Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv")
df_lp_sayur_kec = bersihkan_file_wide(path, nama_value="luas_panen", cek_satuan_konsisten=None)

display(df_lp_sayur_kec.head(10))
df_lp_sayur_kec.to_csv(os.path.join(OUT_HORTI, "luas_panen_hortikultura_sayur_per_kecamatan.csv"),
                       index=False, encoding="utf-8-sig")

In [ ]:
# 9. HORTIKULTURA BUAH — Produksi per Kecamatan (satuan seragam: kuintal)

path = os.path.join(RAW_HORTI, "Produksi_Hortikultura_Buah_per_kecamatan.csv")
df_produksi_buah_kec = bersihkan_file_wide(path, nama_value="produksi_kuintal", cek_satuan_konsisten="Kw")

display(df_produksi_buah_kec.head(10))
df_produksi_buah_kec.to_csv(os.path.join(OUT_HORTI, "produksi_hortikultura_buah_per_kecamatan.csv"),
                            index=False, encoding="utf-8-sig")

In [ ]:
# 10. HORTIKULTURA SAYUR — Produksi per Kecamatan (satuan seragam: kuintal)

path = os.path.join(RAW_HORTI, "Produksi_Hortikultura_Sayur_per_kecamatan.csv")
df_produksi_sayur_kec = bersihkan_file_wide(path, nama_value="produksi_kuintal", cek_satuan_konsisten="Kw")

display(df_produksi_sayur_kec.head(10))
df_produksi_sayur_kec.to_csv(os.path.join(OUT_HORTI, "produksi_hortikultura_sayur_per_kecamatan.csv"),
                             index=False, encoding="utf-8-sig")

In [ ]:
# 11. FILE TINGKAT KABUPATEN (tidak per kecamatan)

path = os.path.join(RAW_HORTI, "Luas_Panen_Hortikultura_Sayur.csv")
df_lp_sayur_kab = bersihkan_file_kabupaten(path, nama_value="luas_panen")
display(df_lp_sayur_kab.head(10))
df_lp_sayur_kab.to_csv(os.path.join(OUT_HORTI, "luas_panen_hortikultura_sayur_kabupaten.csv"),
                       index=False, encoding="utf-8-sig")

path = os.path.join(RAW_HORTI, "Total_Produksi_Hortikultura_Buah.csv")
df_produksi_buah_kab = bersihkan_file_kabupaten(path, nama_value="produksi_kuintal")
display(df_produksi_buah_kab.head(10))
df_produksi_buah_kab.to_csv(os.path.join(OUT_HORTI, "total_produksi_hortikultura_buah_kabupaten.csv"),
                            index=False, encoding="utf-8-sig")

path = os.path.join(RAW_HORTI, "Total_Produksi_Hortikultura_Sayur.csv")
df_produksi_sayur_kab = bersihkan_file_kabupaten(path, nama_value="produksi_kuintal")
display(df_produksi_sayur_kab.head(10))
df_produksi_sayur_kab.to_csv(os.path.join(OUT_HORTI, "total_produksi_hortikultura_sayur_kabupaten.csv"),
                             index=False, encoding="utf-8-sig")

## Ringkasan hal yang perlu kamu tindak lanjuti
1. **Komoditas "Keji Beling"** (biofarmaka) dibuang dari hasil karena isinya `...` di semua kecamatan.
   Kalau datanya memang ada di sumber BPS asli, cek ulang filenya — kemungkinan ada kesalahan saat
   copy-paste/export.
2. **"Serai" (biofarmaka)** dan **"Jamur tiram" (luas panen sayur)** punya label satuan yang tidak
   konsisten antara teks dan kodenya. Cek ke sumber aslinya satuan mana yang benar sebelum dipakai untuk
   hitungan apa pun (mis. produktivitas).
3. **Satuan campuran** (m² vs pohon di biofarmaka; ha vs m² di luas panen sayur) — jangan dijumlahkan
   lintas komoditas tanpa memperhatikan kolom `satuan`.
4. Semua angka "-" sudah dianggap **0** (konvensi BPS). Kalau asumsi ini salah untuk kasusmu, ubah
   `parse_angka()` di cell 4.
5. File kabupaten (`*_kabupaten.csv`) belum dicocokkan nama komoditasnya satu-satu dengan file per
   kecamatan (mis. "Bawang Daun" vs "Bawang Daun/Scallion") — itu perlu tabel master komoditas terpisah,
   belum dibuat di notebook ini.